# Super Factory Ultimate - RunPod Setup & Run\n\nOptimized for **ComfyUI - Blackwell Edition** template (RTX 5090 / B200).\n\nRun each cell in order. Steps 1-3 detect what's already installed and skip accordingly.

## Step 1: Clone & Install

In [ ]:
# Clone the repo (skip if already done)\nimport os\nif not os.path.exists("/workspace/superfactory/main.py"):\n    !cd /workspace && git clone -b claude/create-masterworkflow-QqqTa https://github.com/ChuftyDigital/final_master.git superfactory\n%cd /workspace/superfactory\n!pip install -r requirements.txt -q

In [ ]:
# Check GPU\n!nvidia-smi\nimport torch\nprint(f"\\nPyTorch: {torch.__version__}")\nprint(f"CUDA: {torch.version.cuda}")\nif torch.cuda.is_available():\n    print(f"GPU: {torch.cuda.get_device_name(0)}")\n    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Install PyTorch for Blackwell (only if CUDA < 12.8)

In [ ]:
# Only run this if the cell above shows CUDA < 12.8
# Skip if already on 12.8+
#!pip uninstall -y torch torchvision torchaudio
#!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
#!pip install -U --pre triton

## Step 3: Install ComfyUI + Custom Nodes

In [ ]:
import os\nCOMFYUI_DIR = "/workspace/ComfyUI"\n\n# Blackwell Edition template already has ComfyUI - just check\nif os.path.exists(COMFYUI_DIR):\n    print(f"ComfyUI already installed at {COMFYUI_DIR}")\n    !cd {COMFYUI_DIR} && git pull -q\nelse:\n    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_DIR}\n\n!cd {COMFYUI_DIR} && pip install -r requirements.txt -q\nprint("ComfyUI ready")

In [ ]:
# Install custom nodes
NODES_DIR = f"{COMFYUI_DIR}/custom_nodes"
os.makedirs(NODES_DIR, exist_ok=True)

nodes = [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git", True),
    ("ComfyUI_IPAdapter_plus", "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git", False),
    ("ComfyUI_InstantID", "https://github.com/cubiq/ComfyUI_InstantID.git", False),
    ("ComfyUI-Impact-Pack", "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git", True),
    ("comfyui_controlnet_aux", "https://github.com/Fannovel16/comfyui_controlnet_aux.git", True),
]

for name, url, has_reqs in nodes:
    path = f"{NODES_DIR}/{name}"
    if not os.path.exists(path):
        !git clone --depth 1 {url} {path} -q
    if has_reqs and os.path.exists(f"{path}/requirements.txt"):
        !pip install -r {path}/requirements.txt -q 2>/dev/null
    print(f"  OK: {name}")

print("\nAll custom nodes installed")

## Step 4: Download AI Models (~30GB)
This takes 15-30 min depending on network speed. Progress is shown per model.

In [ ]:
%cd /workspace/superfactory
!python main.py install --comfyui-path /workspace/ComfyUI

## Step 5: Start ComfyUI (runs in background)

In [ ]:
import subprocess, time

# Start ComfyUI as background process
comfy_proc = subprocess.Popen(
    ["python", "main.py",
     "--listen", "0.0.0.0",
     "--port", "8188",
     "--highvram",
     "--fp8_e4m3fn-unet",
     "--bf16-unet",
     "--disable-xformers"],
    cwd=COMFYUI_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("ComfyUI starting (PID: {})".format(comfy_proc.pid))
print("Waiting for server...")

import requests
for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8188/system_stats", timeout=3)
        if r.status_code == 200:
            print(f"ComfyUI is READY on port 8188")
            break
    except:
        pass
    time.sleep(2)
else:
    print("WARNING: ComfyUI may still be loading. Check logs below.")
    # Show last output
    print(comfy_proc.stdout.read(4096).decode(errors='ignore'))

## Step 6: Check Status

In [ ]:
%cd /workspace/superfactory
!python main.py status

## Step 7: Generate Reference Images (63 total, ~15 min)

In [ ]:
!python main.py references

## Step 8: Test Run (5 images, ~1 min)

In [ ]:
!python main.py generate --character 1 --lanes sfw --count 5

## Step 9: Full Production Run (16,800 images)
This runs all 21 characters x 4 lanes x 200 images each.
Estimated time on RTX 5090: ~50-80 hours. Uses resume - safe to interrupt and restart.

In [ ]:
!python main.py generate

---
## Utilities

In [ ]:
# Check how many images have been generated so far
import glob
images = glob.glob("/workspace/superfactory/output/**/*.png", recursive=True)
print(f"Total images generated: {len(images)}")

# Breakdown by character
from collections import Counter
chars = Counter(p.split('/')[4] for p in images if len(p.split('/')) > 4)
for char, count in sorted(chars.items()):
    print(f"  {char}: {count} images")

In [ ]:
# Preview a generated image
from IPython.display import Image, display
import glob

images = sorted(glob.glob("/workspace/superfactory/output/**/*.png", recursive=True))
if images:
    print(f"Showing: {images[0]}")
    display(Image(filename=images[0], width=512))
else:
    print("No images generated yet")

In [ ]:
# Resume generation if interrupted (just run the same command again)
# It automatically picks up where it left off
!python main.py generate

In [ ]:
# Generate specific character only
# Change the number (1-21) to pick a character
!python main.py generate --character 1

In [ ]:
# Generate specific lane only
!python main.py generate --lanes sfw suggestive

In [ ]:
# Export all prompts to text files (for manual ComfyUI use)
!python main.py prompts

In [ ]:
# Stop ComfyUI if needed
try:
    comfy_proc.terminate()
    print("ComfyUI stopped")
except:
    print("ComfyUI was not running")